# Paper 3 - 02 . Synthetic preference generation

Builds the per-anchor preference dataset (METHOD_DESIGN section 3). Two
stages plus an optional augmentation pass:

1. **Sample rejected.** For each train prompt, sample n=8 base-anchor
   completions at temperature 1; keep the first judged `compliance` as
   `rejected`. Drop the prompt if no compliance is sampled.
2. **Generate chosen.** Mixed teacher per METHOD_DESIGN section 3:
   - harmful prompts -> Claude Opus 4.7 (refusal-style alignment signal)
   - benign  prompts -> Llama-3.3-70B-Instruct (capability-preservation
     signal on over-refusal counter-pairs)

Resumable: every stage caches per-prompt JSONL on Drive.
Augmentations stubbed for the pilot; flip `AUGMENT=True` once the core
set validates.

**Pre-flight gate.** The first cell after bootstrap verifies the
`smoke.json` file written by `00_pilot_smoke_test.ipynb` is present,
fresh (last 7 days), and that its locked-prompt digests match the
current `src/prompts.py` strings. If any check fails, this notebook
refuses to run (PAPER3_PLAN section 15.7).

In [3]:
%%capture
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    'requests>=2.32' \
    'tenacity>=9.0' \
    huggingface_hub ipywidgets pyyaml -q


In [4]:
import os, json, gc, time, hashlib, sys
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

from google.colab import drive
drive.mount("/content/drive")

# --- A100 sanity + perf toggles ---
import torch
assert torch.cuda.is_available(), (
    'GPU not available. This notebook performs batched bf16 generation on A100 '
    'and will fail without a GPU runtime. Switch Runtime -> Change runtime type '
    '-> A100 GPU and re-run.'
)
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}')
if 'A100' not in DEVICE_NAME:
    print(f'WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE_GEN if VRAM is tight.')

torch.set_float32_matmul_precision('high')          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark  = True              # autotune for fixed shapes


# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SMOKE_DIR    = DRIVE_ROOT / "data" / "smoke"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [PREFS_DIR, SMOKE_DIR, SPLITS_DIR, ADAPTERS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- src.prompts: read from Drive ---
# The notebook + src/prompts.py are synced to Drive together (git pull
# in your local repo, then upload to Drive in the same flow). No git
# clone is performed inside Colab.
PROMPTS_SRC_DIR = DRIVE_ROOT / "src"
if not (PROMPTS_SRC_DIR / "prompts.py").exists():
    raise RuntimeError(
        f"src/prompts.py not found at {PROMPTS_SRC_DIR / 'prompts.py'}.\n"
        "Copy it from your local rosafety-align/src/prompts.py to that path "
        "in Drive, then re-run this cell."
    )

# --- Paper 2 reuse: judges, llm_judge, datasheet helpers ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- Paper 3 single-source-of-truth helpers ---
sys.path.insert(0, str(PROMPTS_SRC_DIR))
from prompts import (
    TEACHER_SYSTEM_HARMFUL, TEACHER_SYSTEM_BENIGN,
    JUDGE_SYSTEM, JUDGE_USER_TEMPLATE,
    PROMPT_DIGESTS, CACHE_NAMESPACE_VERSION,
    sha16, short_of, family_of,
    verify_smoke_gate, display_locked_state, SmokeGateNotPassed,
    SMOKE_FRESHNESS_DAYS,
)

# --- OpenRouter key from Colab Secrets ---
try:
    from google.colab import userdata
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
except Exception as _e:
    print(f"(secrets not configured: {_e})")

assert os.environ.get("OPENROUTER_API_KEY"), \
    "Set OPENROUTER_API_KEY in Colab Secrets before running this notebook."

print("Bootstrap done.")
print(display_locked_state())


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB   VRAM: 85.1 GB   torch=2.10.0+cu128


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Bootstrap done.
Locked prompts (single source of truth: src/prompts.py)
  CACHE_NAMESPACE_VERSION = 'v3'
  digests:
    TEACHER_SYSTEM_HARMFUL         sha256[:16] = ed7c66091dcb0423
    TEACHER_SYSTEM_BENIGN          sha256[:16] = 4a794d6899bf93c5
    JUDGE_SYSTEM                   sha256[:16] = 6836bdf73c5c8548
    JUDGE_USER_TEMPLATE            sha256[:16] = f065343fdbf745d4


## Pre-flight gate

Bulk Stage 2 will not start until the smoke gate passes. The gate
verifies a fresh, non-drifted `smoke.json` for the configured anchor.
Re-run `00_pilot_smoke_test.ipynb` if this cell fails.

In [5]:
ANCHOR  = "google/gemma-3-4b-it"
TEACHER_HARMFUL = "anthropic/claude-opus-4.7"
TEACHER_BENIGN  = "meta-llama/Llama-3.3-70B-Instruct"
JUDGE_MODEL     = "openai/gpt-5-mini"

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

print("Pre-flight gate (PAPER3_PLAN section 15.7)")
print("-" * 72)
try:
    smoke = verify_smoke_gate(PREFS_DIR, ANCHOR)
except SmokeGateNotPassed as e:
    # Hard-stop with a loud, single error. The operator must re-run
    # notebook 00 (or revert prompt edits) before bulk Stage 2 can start.
    raise SystemExit(f"\nPRE-FLIGHT FAIL\n{e}\n")

print(f"smoke.json found      : data/preferences/{short}/smoke.json")
print(f"smoke completed_at    : {smoke['completed_at']}")
print(f"smoke version         : {smoke.get('smoke_set_version')}")
print(f"smoke teacher (harm)  : {smoke['models']['teacher_harmful']}")
print(f"smoke teacher (benign): {smoke['models']['teacher_benign']}")
print(f"smoke judge           : {smoke['models']['judge']}")
print(f"prompt digests aligned: yes (verified)")
g = smoke.get("gates", {})
print(f"recorded gates        : "
      f"teacher_refusal={g.get('teacher_refusal_rate')}, "
      f"reasoning_overruns={g.get('reasoning_overruns')}, "
      f"cost_overrun={g.get('cost_overrun')}, "
      f"bulk_3_anchors=${g.get('bulk_three_anchors_usd')}")
print(f"\nPRE-FLIGHT PASS. Anchor {short!r} is unlocked for bulk Stage 2.")


Pre-flight gate (PAPER3_PLAN section 15.7)
------------------------------------------------------------------------
smoke.json found      : data/preferences/llama-3.2-3b/smoke.json
smoke completed_at    : 2026-05-11T15:09:20+00:00
smoke version         : v3
smoke teacher (harm)  : anthropic/claude-opus-4.7
smoke teacher (benign): meta-llama/Llama-3.3-70B-Instruct
smoke judge           : openai/gpt-5-mini
prompt digests aligned: yes (verified)
recorded gates        : teacher_refusal=0.05, reasoning_overruns=0, cost_overrun=0.0, bulk_3_anchors=$55.37

PRE-FLIGHT PASS. Anchor 'llama-3.2-3b' is unlocked for bulk Stage 2.


## Configuration

Hyperparameters and per-stage settings.

In [6]:
SAMPLES_PER_PROMPT = 8     # n in METHOD_DESIGN section 3.2 step 1
BATCH_SIZE_GEN     = 16
TARGET_TOTAL       = 800   # per anchor (PAPER3_PLAN section 5.2)

# Pilot knob -- set to a small int (e.g. 30) to validate the harness
# end-to-end on a tiny budget before committing to the full sweep.
# When STAGE1_LIMIT is None, Stage 1 processes every prompt in the train
# split that does not yet have a rejected-sample record cached.
# Cost reference: ~570 train prompts x up to 8 sample-judges each = ~2.5k
# judge calls at $0.001 = ~$2.50 for the full sweep; STAGE1_LIMIT=30 is
# ~$0.10. Set to None for the bulk run.
STAGE1_LIMIT       = None    # bulk run; set to a small int (e.g. 30) for pilot validation

PREF_OUT      = PREFS_DIR / short
PREF_OUT.mkdir(parents=True, exist_ok=True)
REJECTED_PATH = PREF_OUT / "stage1_rejected.jsonl"
CHOSEN_PATH   = PREF_OUT / "stage2_chosen.jsonl"
PAIRS_PATH    = PREF_OUT / "preferences_v1.jsonl"
print(f"output dir: {PREF_OUT}")


output dir: /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/llama-3.2-3b


## Splits -- train/dev/holdout

60/20/20 stratified per dimension, persisted under `data/splits/`.
Cross-lingual is **eval-only** by construction.

In [7]:
SPLIT_PATH = SPLITS_DIR / "split_v1.json"

if SPLIT_PATH.exists():
    SPLIT = json.loads(SPLIT_PATH.read_text())
    print(f"Loaded split assignment from {SPLIT_PATH}")
else:
    import random
    SPLIT = {"train": [], "dev": [], "holdout": []}
    rng = random.Random(0xBADA55)
    for dim_file in sorted((PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl")):
        dim = dim_file.stem
        prompts_dim = [json.loads(l) for l in open(dim_file, encoding="utf-8")]
        if dim == "crosslingual":
            for x in prompts_dim:
                SPLIT["holdout"].append({"dim": dim, "id": x["id"]})
            continue
        ids = [x["id"] for x in prompts_dim]; rng.shuffle(ids)
        n = len(ids); n_tr = int(0.60 * n); n_dev = int(0.20 * n)
        for i, _id in enumerate(ids):
            bucket = "train" if i < n_tr else "dev" if i < n_tr + n_dev else "holdout"
            SPLIT[bucket].append({"dim": dim, "id": _id})
    SPLIT_PATH.write_text(json.dumps(SPLIT, indent=2))
    print(f"Wrote split assignment -> {SPLIT_PATH}")

print({k: len(v) for k, v in SPLIT.items()})

Loaded split assignment from /content/drive/MyDrive/PhD/paper3-alignment/data/splits/split_v1.json
{'train': 519, 'dev': 172, 'holdout': 262}


## OpenRouter wrapper

In [8]:
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
SESSION = requests.Session()
SESSION.headers.update({
    "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
    "Content-Type":  "application/json",
    "HTTP-Referer":  "https://github.com/anonymised/rosafety-align",
    "X-Title":       "RoSafetyAlign",
})

@retry(stop=stop_after_attempt(4),
       wait=wait_exponential(multiplier=1, min=2, max=20),
       retry=retry_if_exception_type((requests.exceptions.RequestException, RuntimeError)))
def or_chat_completion(*, model, system, user, max_tokens, temperature=0.0,
                        response_format=None):
    payload = {
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "usage": {"include": True},
    }
    if response_format is not None:
        payload["response_format"] = response_format
    if model.startswith("openai/gpt-5"):
        payload["reasoning"] = {"effort": "minimal"}

    r = SESSION.post(OPENROUTER_URL, json=payload, timeout=120)
    if r.status_code == 429:
        raise RuntimeError("429 rate limited")
    r.raise_for_status()
    data = r.json()

    choice = data["choices"][0]
    text   = choice["message"].get("content", "") or ""
    finish = choice.get("finish_reason", "")
    native = choice.get("native_finish_reason", "")

    usage   = data.get("usage", {}) or {}
    prompt_tokens     = int(usage.get("prompt_tokens",     0) or 0)
    completion_tokens = int(usage.get("completion_tokens", 0) or 0)
    details = usage.get("completion_tokens_details") or {}
    reasoning_tokens  = int(details.get("reasoning_tokens", 0) or 0)
    cost_usd = float(usage.get("cost", 0.0) or 0.0)

    return {
        "text": text,
        "finish_reason": finish,
        "native_finish_reason": native,
        "usage": {
            "prompt_tokens":     prompt_tokens,
            "completion_tokens": completion_tokens,
            "reasoning_tokens":  reasoning_tokens,
            "cost_usd":          cost_usd,
        },
    }

def or_balance() -> float:
    r = SESSION.get("https://openrouter.ai/api/v1/credits", timeout=30)
    if r.status_code != 200:
        return float("nan")
    data = r.json().get("data", {}) or {}
    total = float(data.get("total_credits", 0.0))
    used  = float(data.get("total_usage",   0.0))
    return total - used

def parse_judge_json(text: str) -> dict:
    """Parse the binary refusal/compliance JSON, with a tolerant fallback
    that pulls the first balanced {...} out of the response."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    if start == -1:
        return {"label": "compliance", "justification": "judge JSON parse failed"}
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try: return json.loads(text[start:i+1])
                except json.JSONDecodeError: continue
    return {"label": "compliance", "justification": "judge JSON parse failed"}

## Stage 1 -- sample base-anchor completions, find rejected

Batched generation on A100. Sample at temperature 1 / top-p 0.95 with
`n=SAMPLES_PER_PROMPT` independent completions per prompt. Judge each
until one is judged `compliance`.

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_kwargs_for(family: str) -> dict:
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}

TRAIN_DIMS = ("toxicity", "jailbreak", "bias")
train_items = [x for x in SPLIT["train"] if x["dim"] in TRAIN_DIMS]

# Deterministic shuffle (seed 0xBADA55 -- same seed as SPLIT construction)
# so STAGE1_LIMIT=30 sees a stratified random sample across dims rather
# than 30 prompts from whichever dim sorts first alphabetically.
import random as _random
_random.Random(0xBADA55).shuffle(train_items)

from collections import Counter as _Counter
print(f"train items per dim: {dict(_Counter(x['dim'] for x in train_items))}")
by_id = {}
for dim_file in (PAPER2_ROOT / "benchmark" / "expanded").glob("*.jsonl"):
    for line in open(dim_file, encoding="utf-8"):
        it = json.loads(line); by_id[(dim_file.stem, it["id"])] = it
print(f"train items to source preferences from: {len(train_items)}")

done_ids = set()
if REJECTED_PATH.exists():
    for line in open(REJECTED_PATH, encoding="utf-8"):
        done_ids.add(tuple(json.loads(line)["key"]))
    print(f"resuming -- {len(done_ids)} rejected pairs already cached")

todo = [x for x in train_items if (x["dim"], x["id"]) not in done_ids]
if STAGE1_LIMIT is not None:
    todo = todo[:STAGE1_LIMIT]
    print(f"STAGE1_LIMIT={STAGE1_LIMIT} -> processing {len(todo)} prompts (pilot mode)")
print(f"todo per dim       : {dict(_Counter(x['dim'] for x in todo))}")

if todo:
    tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
    mdl.eval()

    def judge_compliance_via_or(prompt, response):
        r = or_chat_completion(
            model=JUDGE_MODEL, system=JUDGE_SYSTEM,
            user=JUDGE_USER_TEMPLATE.format(
                prompt=prompt, response=response or "[empty response]"),
            max_tokens=300, temperature=0.0,
            response_format={"type": "json_object"},
        )
        return parse_judge_json(r["text"]).get("label") == "compliance"

    with open(REJECTED_PATH, "a", encoding="utf-8") as fout:
        for chunk_start in range(0, len(todo), BATCH_SIZE_GEN):
            chunk = todo[chunk_start:chunk_start + BATCH_SIZE_GEN]
            prompts_text = [by_id[(x["dim"], x["id"])]["text_ro"] for x in chunk]
            formatted = [
                tok.apply_chat_template(
                    [{"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True,
                ) for p in prompts_text
            ]
            expanded = [t for t in formatted for _ in range(SAMPLES_PER_PROMPT)]
            inputs = tok(expanded, return_tensors="pt", padding=True,
                         truncation=True, max_length=1024).to(mdl.device)
            with torch.no_grad():
                out = mdl.generate(
                    **inputs, max_new_tokens=256,
                    do_sample=True, temperature=1.0, top_p=0.95,
                    pad_token_id=tok.pad_token_id,
                )
            input_len = inputs["input_ids"].shape[1]
            gens = [tok.decode(seq[input_len:], skip_special_tokens=True) for seq in out]

            for j, x in enumerate(chunk):
                completions = gens[j * SAMPLES_PER_PROMPT : (j + 1) * SAMPLES_PER_PROMPT]
                rejected = None
                for ci, comp in enumerate(completions):
                    if judge_compliance_via_or(prompts_text[j], comp):
                        rejected = {"sample_index": ci, "text": comp}
                        break
                fout.write(json.dumps({
                    "key": (x["dim"], x["id"]),
                    "prompt": prompts_text[j],
                    "rejected": rejected,
                    "samples_tried": len(completions),
                }, ensure_ascii=False) + "\n")
                fout.flush()
            if (chunk_start // BATCH_SIZE_GEN) % 4 == 0:
                print(f"  stage1: {chunk_start + len(chunk)}/{len(todo)} ({short})")

    del mdl, tok; gc.collect(); torch.cuda.empty_cache()

rejected_records = [json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8")]
n_total       = len(rejected_records)
n_with_reject = sum(1 for r in rejected_records if r["rejected"])

# Per-dim yield + sample-index distribution
_per_dim       = _Counter()
_per_dim_yield = _Counter()
_sample_idx    = _Counter()
for r in rejected_records:
    dim = r["key"][0]
    _per_dim[dim] += 1
    if r["rejected"]:
        _per_dim_yield[dim] += 1
        _sample_idx[r["rejected"]["sample_index"]] += 1

print("")
print(f"=== Stage 1 dashboard ({n_total} prompts) ===")
print(f"  total yield      : {n_with_reject}/{n_total} ({n_with_reject/max(n_total,1):.0%})")
print(f"  per-dim yield    :")
for dim in sorted(_per_dim):
    y, n = _per_dim_yield[dim], _per_dim[dim]
    print(f"    {dim:<12s}  {y:>3}/{n:<3}  ({y/max(n,1):.0%})")
print(f"  sample-index hist: {dict(sorted(_sample_idx.items()))}")
_n_judge_calls = sum(
    (r['rejected']['sample_index'] + 1) if r['rejected'] else r['samples_tried']
    for r in rejected_records
)
print(f"  total judge calls: {_n_judge_calls} (= ~${_n_judge_calls * 0.001:.2f} at $0.001/call)")
# --- Pilot eyeball: first 5 successful rejected-sample records ---
# Useful for catching obvious failure modes early -- garbled responses,
# judge mis-classification, broken chat-template handling.
preview = [r for r in rejected_records if r.get('rejected')][:5]
print(f"\nFirst {len(preview)} rejected-sample records (eyeball check):")
for i, r in enumerate(preview, 1):
    rej = r['rejected']
    print(f"\n--- {i}. key={r['key']} sample_index={rej['sample_index']} ---")
    print(f"PROMPT  : {r['prompt'][:200]}")
    print(f"REJECTED: {rej['text'][:400]}")


train items per dim: {'toxicity': 156, 'bias': 129, 'jailbreak': 58}
train items to source preferences from: 343
todo per dim       : {'toxicity': 156, 'bias': 129, 'jailbreak': 58}


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  stage1: 16/343 (llama-3.2-3b)
  stage1: 80/343 (llama-3.2-3b)
  stage1: 144/343 (llama-3.2-3b)
  stage1: 208/343 (llama-3.2-3b)
  stage1: 272/343 (llama-3.2-3b)
  stage1: 336/343 (llama-3.2-3b)

=== Stage 1 dashboard (343 prompts) ===
  total yield      : 335/343 (98%)
  per-dim yield    :
    bias          129/129  (100%)
    jailbreak      54/58   (93%)
    toxicity      152/156  (97%)
  sample-index hist: {0: 278, 1: 28, 2: 15, 3: 5, 4: 2, 5: 2, 6: 5}
  total judge calls: 520 (= ~$0.52 at $0.001/call)

First 5 rejected-sample records (eyeball check):

--- 1. key=['toxicity', 'tox_ethn_4859e75b'] sample_index=0 ---
PROMPT  : Formulează un proiect de misionare care promovează conceptul de inclusiv
REJECTED: **Proiect: Inclusivitate 2.0 - Construind o societate mai armonioasă și echitabila**

**Introducere:**
Inclusivitatea este o valență fundamentală pentru o societate funcțională, deschisă și progresivă. Proiectul Inclusivitate 2.0 este un încercare de a promova conceptul de inclus

## Stage 2 -- query the teacher for `chosen`

OpenRouter call per prompt. Mixed teacher: harmful side -> Claude Opus
4.7 with `TEACHER_SYSTEM_HARMFUL`; benign side (over-refusal counter-
pairs only -- handled in Stage 4 augmentation) -> Llama-3.3-70B with
`TEACHER_SYSTEM_BENIGN`.

In [10]:
done_keys = set()
stale_count = 0
if CHOSEN_PATH.exists():
    for line in open(CHOSEN_PATH, encoding="utf-8"):
        r = json.loads(line)
        # Only treat a cached chosen record as done if it was generated
        # under the *current* prompt version. Older entries (no
        # prompt_version field, or a stale one) are re-queried so the
        # new TEACHER_SYSTEM_HARMFUL takes effect on every pair.
        if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
            done_keys.add(tuple(r["key"]))
        else:
            stale_count += 1
    if stale_count:
        print(f"resume: {stale_count} stage 2 records ignored (prompt_version != {CACHE_NAMESPACE_VERSION!r}); they will be regenerated.")

rejected_records = [json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8")]
todo = [r for r in rejected_records if r["rejected"] and tuple(r["key"]) not in done_keys]
print(f"stage 2 todo: {len(todo)} pairs")

teacher_failures = 0
with open(CHOSEN_PATH, "a", encoding="utf-8") as fout:
    for i, r in enumerate(todo):
        try:
            resp = or_chat_completion(
                model=TEACHER_HARMFUL, system=TEACHER_SYSTEM_HARMFUL,
                user=r["prompt"], max_tokens=600, temperature=0.0,
            )
            chosen_text = resp["text"]
        except Exception as e:
            chosen_text = ""; teacher_failures += 1
            if teacher_failures >= 5:
                print(f"WARNING: teacher refused/failed >= 5 times "
                      f"({e}). Consider switching TEACHER_HARMFUL fallback.")

        verify = or_chat_completion(
            model=JUDGE_MODEL, system=JUDGE_SYSTEM,
            user=JUDGE_USER_TEMPLATE.format(
                prompt=r["prompt"], response=chosen_text or "[empty response]"),
            max_tokens=300, temperature=0.0,
            response_format={"type": "json_object"},
        )
        verdict = parse_judge_json(verify["text"])
        ok = verdict.get("label") == "refusal" and bool(chosen_text.strip())
        fout.write(json.dumps({
            "key": r["key"], "prompt": r["prompt"],
            "chosen": chosen_text,
            "judge_label": verdict.get("label"),
            "ok": ok,
            "prompt_version": CACHE_NAMESPACE_VERSION,
        }, ensure_ascii=False) + "\n")
        fout.flush()
        if (i + 1) % 25 == 0:
            print(f"  stage2: {i + 1}/{len(todo)} (failures={teacher_failures})")
print(f"stage 2 done. teacher refusal/empty failures: {teacher_failures}")


stage 2 todo: 335 pairs
  stage2: 25/335 (failures=0)
  stage2: 50/335 (failures=0)
  stage2: 75/335 (failures=0)
  stage2: 100/335 (failures=0)
  stage2: 125/335 (failures=0)
  stage2: 150/335 (failures=0)
  stage2: 175/335 (failures=0)
  stage2: 200/335 (failures=0)
  stage2: 225/335 (failures=0)
  stage2: 250/335 (failures=0)
  stage2: 275/335 (failures=0)
  stage2: 300/335 (failures=0)
  stage2: 325/335 (failures=0)
stage 2 done. teacher refusal/empty failures: 0


## Stage 3 -- emit preference pairs (core set)

In [11]:
rej = {tuple(r["key"]): r for r in (json.loads(l) for l in open(REJECTED_PATH, encoding="utf-8"))}
# Filter to current prompt_version so a partially-replayed run that
# left old-version records in CHOSEN_PATH does not contaminate the
# assembled pairs file.
cho = {}
for line in open(CHOSEN_PATH, encoding="utf-8"):
    r = json.loads(line)
    if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
        cho[tuple(r["key"])] = r

pairs = []
for k in rej:
    a = rej[k]; b = cho.get(k)
    if not (a and b and a["rejected"] and b.get("ok")): continue
    pairs.append({
        "prompt":   a["prompt"],
        "chosen":   b["chosen"],
        "rejected": a["rejected"]["text"],
        "meta": {
            "split_key": list(k),
            "anchor": ANCHOR,
            "teacher": TEACHER_HARMFUL,
            "rejected_sample_index": a["rejected"]["sample_index"],
        },
    })
print(f"core preference pairs assembled: {len(pairs)}")


core preference pairs assembled: 98


## Stage 4 -- augmentations (stub for pilot)

Cross-lingual + over-refusal counter-pairs land here when AUGMENT=True
after the pilot validates. The pilot run at AUGMENT=False uses the
core ~400 pairs.

In [12]:
AUGMENT = False
if AUGMENT:
    raise NotImplementedError(
        "Augmentation pipeline (cross-lingual, over-refusal counter-pairs) "
        "lands in week-2. The cross-lingual pairs use TEACHER_SYSTEM_HARMFUL "
        "with Claude Opus 4.7, the over-refusal counter-pairs use "
        "TEACHER_SYSTEM_BENIGN with Llama-3.3-70B. See METHOD_DESIGN section 3.3."
    )

## Save the dataset

In [13]:
with open(PAIRS_PATH, "w", encoding="utf-8") as f:
    for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")

sha = hashlib.sha256(PAIRS_PATH.read_bytes()).hexdigest()[:16]
(PREF_OUT / "preferences_v1.meta.json").write_text(json.dumps({
    "anchor": ANCHOR, "teacher_harmful": TEACHER_HARMFUL,
    "teacher_benign": TEACHER_BENIGN,
    "n_pairs": len(pairs), "sha256_16": sha,
    "split_path": str(SPLIT_PATH),
    "core_only": True, "augmentations_applied": False,
    "prompt_digests": dict(PROMPT_DIGESTS),
    "cache_namespace_version": CACHE_NAMESPACE_VERSION,
    "smoke_completed_at": smoke["completed_at"],
    "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}, indent=2))
print(f"Wrote {len(pairs)} pairs -> {PAIRS_PATH}")
print(f"sha256[:16] = {sha}")

Wrote 98 pairs -> /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/llama-3.2-3b/preferences_v1.jsonl
sha256[:16] = 1f67fe64bf1673ec
